In [ ]:
# Global Oil and Gas Exposure Study
# 01: PopExposure Analysis

# This file applies Popexposure to finds the number of people living near buffered oil and gas wells. 
# Author: Lara Schwarz
# Last updated: September 2025

## Function step 1: Estimating population affected by oil and gas wells - country level 
## Function step 2: Estimating population affected by oil and gas wells by adm2 
## Function step 3: Estimating population affected by oil and gas wells by adm2 - out of country wells
## Function step 4: Estimating population affected by oil and gas wells by adm2 - offshore wells
## Function step 5: Estimating total area exposed by country 
## Function step 6: Estimating ogd well-weighted population exposure 
## Function step 7: Estimating average GRDI by adm2

In [6]:
## Importing libraries

import geopandas as gpd
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
from thefuzz import process
import pandas as pd
import pathlib
import pandas as pd
import sys
import os
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
import geopandas as gpd
from rasterio.plot import show
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import fiona
import geopandas as gpd
from shapely.geometry import Polygon
import ee
import geemap
import matplotlib.cm as cm
import matplotlib.colors as colors
from shapely.geometry import box
import math
from shapely.geometry import Point
import glob
import seaborn as sns
import openpyxl
import subprocess
from pathlib import Path
from popexposure import PopEstimator



In [7]:
# Define base paths
repo = pathlib.Path.cwd().parent.parent
share_path = pathlib.Path("/Users/larasch/Library/CloudStorage/OneDrive-SharedLibraries-UW/casey_cohort - Documents/data")

# Define code path and add to sys.path
code_path = repo / "code"

# Define data paths
# location of the oil and gas data
oil_path = share_path / "environmental/oil_and_gas/raw_data/ogim.parquet"

# location of the adminstrative boundary shapefile
countries_path = share_path / "geo_boundaries/processed_data/country_admin/country_geom_filtered.parquet"

# location of global shapefile
shapefile_path = repo / "data/OneDrive_1_12-11-2024/lbd_standard_admin_2.shp"

# location of the population data
pop_path = share_path / "social_including_census/raw_data/GHS_POP_E2020_GLOBE_R2023A_54009_100_V1_0/GHS_POP_E2020_GLOBE_R2023A_54009_100_V1_0.tif"

# Define the directory for the country-specific parquet files
country_dir = share_path / "geo_boundaries/processed_data/countries_ogd"

# Define the directory for the OGIM country-specific parquet files
ogim_dir = share_path / "environmental/oil_and_gas/raw_data/ogim_by_country"

# Define the directory for results
results_dir = repo / "results"

# Import local modules
from popexposure import *

# Use estimator
pop_est = PopEstimator()

In [ ]:
# Function step 1: Estimating population affected by oil and gas wells - country level 

## Apply THE function
results = {}

# List of countries to loop through
country_list = [
    "argentina", "australia", "austria", "belgium", "bolivia", "brazil", "canada", "chile", 
    "colombia", "denmark", "ecuador", "ethiopia", "france", "germany", "ireland", "italy", 
    "libya", "mexico", "mozambique", "netherlands", "new_zealand", "norway", 
    "papua_new_guinea", "paraguay", "peru", "saudi_arabia", "south_sudan", "sudan", 
    "united_arab_emirates", "united_kingdom", "united_states", "venezuela"
]

# Find total num ppl residing 
exposed_list = []

# Loop through each country
for country in country_list:
    print(f"Processing {country}...")

    # types of wells
    hazard_paths = oil_dir / f"OGIM_{country}.parquet"
    
    # country parquet
    admin_units_path = country_dir / f"{country}.parquet"

    # Prepare admin units data
    admin_units = pop_est.prep_data(admin_units_path, geo_type='spatial_unit')

    
    for hazard_path in [hazard_paths]:
        # Prepare hazard data for this year
        hazards = pop_est.prep_data(hazard_path, geo_type='hazard')
        # Estimate exposed population
        exposed = pop_est.exposed_pop(
            pop_path=pop_path,
            hazard_specific=False,  # set to True if you want per-hazard results
            hazards=hazards,
            spatial_units=admin_units
        )
        exposed_list.append(exposed)

# Estimate total population per spatial unit for this country
    pop_df = pop_est.pop(pop_path=pop_path, spatial_units=admin_units)
    pop_df["country"] = country  # Optionally add country column for tracking
    pop_list.append(pop_df)      # <-- Collect each country's pop_df

# Combine all countries
all_exposed_df = pd.concat(exposed_list, axis=0, ignore_index=True)
all_pop_df = pd.concat(pop_list, axis=0, ignore_index=True)

# Merge exposed and total population on ID_spatial_unit and country
combined_df = all_exposed_df.merge(
    all_pop_df[["ID_spatial_unit", "population"]],
    on=["ID_spatial_unit"],
    how="left"
)

# Save output
output_path = results_dir / "ogd_country_pop_exposed.csv"
combined_df.to_csv(output_path, index=False)
print(f"✅ Saved: {output_path}")

In [ ]:
# Function step 2: Estimating population affected by oil and gas wells by adm2 

## Apply THE function
results = {}

# List of countries to loop through
country_list = [
    "argentina", "australia", "austria", "belgium", "bolivia", "brazil", "canada", "chile", 
    "colombia", "denmark", "ecuador", "ethiopia", "france", "germany", "ireland", "italy", 
    "libya", "mexico", "mozambique", "netherlands", "new_zealand", #"norway", 
    "papua_new_guinea", "paraguay", "peru", "saudi_arabia", "south_sudan", "sudan", 
    "united_arab_emirates", "united_kingdom", "united_states", "venezuela"
]

# Use estimator
pop_est = PopEstimator()

# Find total num ppl residing 
exposed_list = []
pop_list = []

# Loop through each country
for country in country_list:
    print(f"Processing {country}...")

    # types of wells
    hazard_paths = ogim_dir / f"OGIM_{country}.parquet"
    
    # country parquet
    admin_units_path = country_dir / f"{country}_adm2.parquet"

    # Prepare admin units data
    admin_units = pop_est.prep_data(admin_units_path, geo_type='admin_unit')

    
    for hazard_path in [hazard_paths]:
        # Prepare hazard data for this year
        hazards = pop_est.prep_data(hazard_path, geo_type='hazard')
        # Estimate exposed population
        exposed = pop_est.est_exposed_pop(
            pop_path=pop_path,
            hazard_specific=False,  # set to True if you want per-hazard results
            hazards=hazards,
            admin_units=admin_units
        )
        exposed["country"] = country  
        exposed_list.append(exposed)

# Estimate total population per spatial unit for this country
    pop_df = pop_est.est_pop(pop_path=pop_path, admin_units=admin_units)
    pop_df["country"] = country  # Optionally add country column for tracking
    pop_list.append(pop_df)      # <-- Collect each country's pop_df

# Combine all countries
all_exposed_df = pd.concat(exposed_list, axis=0, ignore_index=True)
all_pop_df = pd.concat(pop_list, axis=0, ignore_index=True)

# Merge exposed and total population on ID_spatial_unit and country
combined_df = all_pop_df.merge(
    all_exposed_df[["admin_unit", "exposed_1km", "exposed_3km","exposed_5km", "country"]],
    on=["admin_unit", "country"],
    how="left"
)

# Add percent exposed columns
combined_df["pct_exposed_1km"] = combined_df["exposed_1km"] / combined_df["population"]
combined_df["pct_exposed_3km"] = combined_df["exposed_3km"] / combined_df["population"]
combined_df["pct_exposed_5km"] = combined_df["exposed_5km"] / combined_df["population"]

# Optionally, handle division by zero or NaN
combined_df["pct_exposed_1km"] = combined_df["pct_exposed_1km"].replace([np.inf, -np.inf], np.nan)
combined_df["pct_exposed_3km"] = combined_df["pct_exposed_3km"].replace([np.inf, -np.inf], np.nan)
combined_df["pct_exposed_5km"] = combined_df["pct_exposed_5km"].replace([np.inf, -np.inf], np.nan)


# Save output
output_path = results_dir / "ogd_adm2_pop_exposed.csv"
combined_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")


In [ ]:
# Function step 3: Estimating population affected by oil and gas wells by adm2 - out of country wells

## Apply THE function
results = {}

# List of countries to loop through
country_list = [
    "argentina", "australia", "austria", "belgium", "bolivia", "brazil", "canada", "chile", 
    "colombia",  "ecuador", "ethiopia", "france", "germany", "ireland", "italy", #"denmark",
    "libya", "mexico", "mozambique", "netherlands", "new_zealand", #"norway", 
    "papua_new_guinea",  "peru", "saudi_arabia", "south_sudan", "sudan", #"paraguay",
    "united_arab_emirates", "united_kingdom", "united_states", "venezuela"
]

# Find total num ppl residing 
exposed_list = []
pop_list = []

# Loop through each country
for country in country_list:
    print(f"Processing {country}...")

    # types of wells
    hazard_paths = oil_dir / f"OGIM_{country}_out_of_country_modis.parquet"

    # country parquet
    admin_units_path = country_dir / f"{country}_adm2.parquet"


    # Check if both files exist
    if not hazard_paths.exists():
        print(f"  Skipping {country}: out of country file not found.")
        continue
    if not admin_units_path.exists():
        print(f"  Skipping {country}: admin2 file not found.")
        continue

    # Prepare admin units data
    admin_units = pop_est.prep_data(admin_units_path, geo_type='admin_unit')

    for hazard_path in [hazard_paths]:
        # Prepare hazard data for this year
        hazards = pop_est.prep_data(hazard_path, geo_type='hazard')
        # Estimate exposed population
        exposed = pop_est.est_exposed_pop(
            pop_path=pop_path,
            hazard_specific=False,  # set to True if you want per-hazard results
            hazards=hazards,
            admin_units=admin_units
        )
        exposed["country"] = country  # <-- Add this line
        exposed_list.append(exposed)

# Estimate total population per spatial unit for this country
    pop_df = pop_est.est_pop(pop_path=pop_path, admin_units=admin_units)
    pop_df["country"] = country  # Optionally add country column for tracking
    pop_list.append(pop_df)      # <-- Collect each country's pop_df

# Combine all countries
all_exposed_df = pd.concat(exposed_list, axis=0, ignore_index=True)
#all_exposed_df = all_exposed_df.groupby(["ID_spatial_unit", "country"], as_index=False).sum(numeric_only=True)
all_pop_df = pd.concat(pop_list, axis=0, ignore_index=True)

# Merge exposed and total population on ID_spatial_unit and country
combined_df = all_pop_df.merge(
    all_exposed_df[["ID_spatial_unit", "exposed_1km", "exposed_3km", "country"]],
    on=["ID_spatial_unit", "country"],
    how="left"
)

# Add percent exposed columns
combined_df["pct_exposed_1km"] = combined_df["exposed_1km"] / combined_df["population"]
combined_df["pct_exposed_3km"] = combined_df["exposed_3km"] / combined_df["population"]

# Optionally, handle division by zero or NaN
combined_df["pct_exposed_1km"] = combined_df["pct_exposed_1km"].replace([np.inf, -np.inf], np.nan)
combined_df["pct_exposed_3km"] = combined_df["pct_exposed_3km"].replace([np.inf, -np.inf], np.nan)

# Group and sum by country and well_country if present
group_cols = ["country"]
if "well_country" in combined_df.columns:
    group_cols.append("well_country")

summary_df = combined_df.groupby(
    group_cols, as_index=False
)[["exposed_1km", "exposed_3km", "population"]].sum(numeric_only=True)

combined_df = combined_df.rename(columns={
    "exposed_1km": "out_of_country_exposed_1km",
    "exposed_3km": "out_of_country_exposed_3km"
})

print(combined_df.head())

# Export summary_df as CSV to the results folder
output_path = results_dir / "ogd_country_pop_exposed_out_of_country.csv"
combined_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

In [ ]:
# Function step 4: Estimating population affected by oil and gas wells by adm2 - offshore wells

## Apply THE function
results = {}

# List of countries to loop through
country_list = [
    "argentina", "australia", "austria", "belgium", "bolivia", "brazil", "canada", "chile", 
    "colombia",  "ecuador", "ethiopia", "france", "germany", "ireland", "italy", "denmark",
    "libya", "mexico", "mozambique", "netherlands", "new_zealand", "norway", 
    "papua_new_guinea",  "peru", "saudi_arabia", "south_sudan", "sudan", "paraguay",
    "united_arab_emirates", "united_kingdom", "united_states", "venezuela"
]

# Find total num ppl residing 
exposed_list = []
pop_list = []

# Loop through each country
for country in country_list:
    print(f"Processing {country}...")

    # types of wells
    hazard_paths = ogim_dir / f"OGIM_{country}_offshore_modis.parquet"

    # country parquet
    admin_units_path = country_dir / f"{country}_adm2.parquet"


    # Check if both files exist
    if not hazard_paths.exists():
        print(f"  Skipping {country}: offshore file not found.")
        continue
    if not admin_units_path.exists():
        print(f"  Skipping {country}: admin2 file not found.")
        continue

    # Prepare admin units data
    admin_units = pop_est.prep_data(admin_units_path, geo_type='admin_unit')

    for hazard_path in [hazard_paths]:
        # Prepare hazard data for this year
        hazards = pop_est.prep_data(hazard_path, geo_type='hazard')
        # Estimate exposed population
        exposed = pop_est.est_exposed_pop(
            pop_path=pop_path,
            hazard_specific=False,  # set to True if you want per-hazard results
            hazards=hazards,
            admin_units=admin_units
        )
        exposed["country"] = country  # <-- Add this line
        exposed_list.append(exposed)

# Estimate total population per spatial unit for this country
    pop_df = pop_est.est_pop(pop_path=pop_path, admin_units=admin_units)
    pop_df["country"] = country  # Optionally add country column for tracking
    pop_list.append(pop_df)      # <-- Collect each country's pop_df

# Combine all countries
all_exposed_df = pd.concat(exposed_list, axis=0, ignore_index=True)
#all_exposed_df = all_exposed_df.groupby(["ID_spatial_unit", "country"], as_index=False).sum(numeric_only=True)
all_pop_df = pd.concat(pop_list, axis=0, ignore_index=True)

# Merge exposed and total population on ID_spatial_unit and country
combined_df = all_pop_df.merge(
    all_exposed_df[["ID_spatial_unit", "exposed_1km", "exposed_3km", "exposed_5km", "country"]],
    on=["ID_spatial_unit", "country"],
    how="left"
)

# Add percent exposed columns
combined_df["pct_exposed_1km"] = combined_df["exposed_1km"] / combined_df["population"]
combined_df["pct_exposed_3km"] = combined_df["exposed_3km"] / combined_df["population"]
combined_df["pct_exposed_5km"] = combined_df["exposed_5km"] / combined_df["population"]

# Optionally, handle division by zero or NaN
combined_df["pct_exposed_1km"] = combined_df["pct_exposed_1km"].replace([np.inf, -np.inf], np.nan)
combined_df["pct_exposed_3km"] = combined_df["pct_exposed_3km"].replace([np.inf, -np.inf], np.nan)
combined_df["pct_exposed_5km"] = combined_df["pct_exposed_5km"].replace([np.inf, -np.inf], np.nan)

# Group and sum by country and well_country if present
group_cols = ["country"]
if "well_country" in combined_df.columns:
    group_cols.append("well_country")

summary_df = combined_df.groupby(
    group_cols, as_index=False
)[["exposed_1km", "exposed_3km", "exposed_5km", "population"]].sum(numeric_only=True)

combined_df = combined_df.rename(columns={
    "exposed_1km": "offshore_exposed_1km",
    "exposed_3km": "offshore_exposed_3km",
    "exposed_5km": "offshore_exposed_5km"
})

print(combined_df.head())

# Export summary_df as CSV to the results folder
output_path = results_dir / "ogd_country_pop_exposed_offshore.csv"
combined_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

In [ ]:
# Function step 5: Estimating total area exposed by country 

# List of countries to loop through
country_list = [
    "argentina", "australia", "austria", "belgium", "bolivia", "brazil", "canada", "chile", 
    "colombia", "denmark", "ecuador", "ethiopia", "france", "germany", "ireland", "italy", 
    "libya", "mexico", "mozambique", "netherlands", "new_zealand", #"norway", 
    "papua_new_guinea", "paraguay", "peru", "saudi_arabia", "south_sudan", "sudan", 
    "united_arab_emirates", "united_kingdom", "united_states", "venezuela"
]

# Find total num ppl residing 
exposed_list = []
pop_list = []  

# Loop through each country
for country in country_list:
    print(f"Processing {country}...")

    # types of wells
    hazard_paths = oil_dir / f"OGIM_{country}.parquet"
    
    # country parquet
    admin_units_path = country_dir / f"{country}_adm2.parquet"

    # Prepare admin units data
    admin_units = pop_est.prep_data(admin_units_path, geo_type='admin_unit')

    
    for hazard_path in [hazard_paths]:
        # Prepare hazard data for this year
        hazards = pop_est.prep_data(hazard_path, geo_type='hazard')
        # Estimate exposed population
        exposed = pop_est.est_exposed_pop(
            pop_path=pop_path,
            hazard_specific=False,  # set to True if you want per-hazard results
            hazards=hazards,
            admin_units=admin_units  # leave as None for estimates not by admin unit
        )
        exposed["country"] = country  # Optionally add country column for tracking

        exposed_list.append(exposed)

# Estimate total population per spatial unit for this country
    pop_df = pop_est.est_pop(pop_path=pop_path, admin_units=admin_units)
    pop_df["country"] = country  # Optionally add country column for tracking
    pop_list.append(pop_df)      # <-- Collect each country's pop_df

# Combine all countries
all_exposed_df = pd.concat(exposed_list, axis=0, ignore_index=True)
all_pop_df = pd.concat(pop_list, axis=0, ignore_index=True)

# Merge exposed and total population on ID_spatial_unit and country
combined_df = all_pop_df.merge(
    all_exposed_df[["ID_admin_unit", "population"]],
    on=["ID_admin_unit"],
    how="left"
)
# Add percent exposed columns
combined_df["pct_exposed_1km"] = combined_df["exposed_1km"] / combined_df["population"]
combined_df["pct_exposed_3km"] = combined_df["exposed_3km"] / combined_df["population"]
combined_df["pct_exposed_5km"] = combined_df["exposed_5km"] / combined_df["population"]

# Optionally, handle division by zero or NaN
combined_df["pct_exposed_1km"] = combined_df["pct_exposed_1km"].replace([np.inf, -np.inf], np.nan)
combined_df["pct_exposed_3km"] = combined_df["pct_exposed_3km"].replace([np.inf, -np.inf], np.nan)
combined_df["pct_exposed_5km"] = combined_df["pct_exposed_5km"].replace([np.inf, -np.inf], np.nan)

# Export summary_df as CSV to the results folder
output_path = results_dir / "ogd_land_area_exposed.csv"
combined_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

In [ ]:
# Function step 6: Estimating ogd well-weighted population exposure 

## Apply THE function
results = {}

# List of countries to loop through
country_list = [
    "argentina", "australia", "austria", "belgium", "bolivia", "brazil", "canada", "chile", 
    "colombia", "denmark", "ecuador", "ethiopia", "france", "germany", "ireland", "italy", 
    "libya", "mexico", "mozambique", "netherlands", "new_zealand", # "norway", 
    "papua_new_guinea", "paraguay", "peru", "saudi_arabia", "south_sudan", "sudan", 
    "united_arab_emirates", "united_kingdom","venezuela" # "united_states", 
]

# Find total num ppl residing 
exposed_list = []
pop_list = []

# Loop through each country
for country in country_list:
    print(f"Processing {country}...")

    # types of wells
    hazard_paths = ogim_dir / f"OGIM_{country}.parquet"
    
    # country parquet
    #admin_units_path = countries_path / f"{country}.parquet"
    admin_units_path = country_dir / f"{country}_adm2.parquet"

    # Prepare admin units data
    admin_units = pop_est.prep_data(admin_units_path, geo_type='admin_unit')

    
    for hazard_path in [hazard_paths]:
        # Prepare hazard data for this year
        hazards = pop_est.prep_data(hazard_path, geo_type='hazard')
        # Estimate exposed population
        exposed = pop_est.est_exposed_pop(
            pop_path=pop_path,
            hazard_specific=True,  # set to True if you want per-hazard results
            hazards=hazards,
            admin_units=admin_units
        )
        exposed_list.append(exposed)

# Estimate total population per spatial unit for this country
    pop_df = pop_est.est_pop(pop_path=pop_path, admin_units=admin_units)
    pop_df["country"] = country  # Optionally add country column for tracking
    pop_list.append(pop_df)      # <-- Collect each country's pop_df

# Combine all countries
all_exposed_df = pd.concat(exposed_list, axis=0, ignore_index=True)
all_pop_df = pd.concat(pop_list, axis=0, ignore_index=True)

# Rename population column in pop_df to avoid confusion
all_pop_df = all_pop_df.rename(columns={"population": "total_population"})

# Merge on ID_admin_unit and country
combined_df = all_exposed_df.merge(
    all_pop_df[["ID_admin_unit", "country", "total_population"]],
    on=["ID_admin_unit", "country"],
    how="left"
)

# Group by country and ID_admin_unit, summing the hazard-specific and total populations
summary_df = combined_df.groupby(["country", "ID_admin_unit"], as_index=False).agg({
    "exposed_population": "sum",
    "total_population": "first"  # Use 'first' because total pop is the same per admin unit
})

print(summary_df.head())

# Save output
output_path = results_dir / "ogd_country_pop_exposed_well_weighted.csv"
combined_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

In [ ]:
# Function step 7: Estimating average GRDI by adm2

# List of countries to loop through
country_list = [
    "argentina", "australia", "austria", "belgium", "bolivia", "brazil", "canada", "chile", 
    "colombia", "denmark", "ecuador", "ethiopia", "france", "germany", "ireland", "italy", 
    "libya", "mexico", "mozambique", "netherlands", "new_zealand", #"norway", 
    "papua_new_guinea", "paraguay", "peru", "saudi_arabia", "south_sudan", "sudan", 
    "united_arab_emirates", "united_kingdom", "united_states", "venezuela"
]

# Define the base paths for GRDI data, this is not population data but has to be named that way for function
pop_path = share_path / "social_including_census/raw_data/GRDI_v1/ciesin_nasa_povmap-grdi-v1.tif"

# Find total num ppl residing 
exposed_list = []
pop_list = []  

# Loop through each country
for country in country_list:
    print(f"Processing {country}...")

    # types of wells
    hazard_paths = ogim_dir / f"OGIM_{country}.parquet"
    
    # country parquet
    admin_units_path = country_dir / f"{country}_adm2.parquet"

    # Prepare admin units data
    admin_units = pop_est.prep_data(admin_units_path, geo_type='admin_unit')

    
    for hazard_path in [hazard_paths]:
        # Prepare hazard data for this year
        hazards = pop_est.prep_data(hazard_path, geo_type='hazard')
        # Estimate exposed population
        exposed = pop_est.est_exposed_pop(
            pop_path=pop_path,
            hazard_specific=False,  # set to True if you want per-hazard results
            hazards=hazards,
            admin_units=admin_units,  # leave as None for estimates not by admin unit
            stat= "mean"    # get average GRDI score for admin 2 unit
        )
        exposed["country"] = country
        exposed_list.append(exposed)

# Estimate total population per spatial unit for this country
   # pop_df = pop_est.pop(pop_path=pop_path, spatial_units=admin_units)
   # pop_df["country"] = country  # Optionally add country column for tracking
    #pop_list.append(pop_df)      # <-- Collect each country's pop_df

# Combine all countries
all_exposed_df = pd.concat(exposed_list, axis=0, ignore_index=True)
#all_pop_df = pd.concat(pop_list, axis=0, ignore_index=True)

all_exposed_df = all_exposed_df.rename(columns={
    'exposed_1km': 'GRDI_1km',
    'exposed_3km': 'GRDI_3km',
    'exposed_5km': 'GRDI_5km'
})

# Save output
# Save output
output_path = results_dir / "ogd_adm2_avg_GRDI.csv"
all_exposed_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Processing argentina...


/opt/anaconda3/envs/pop_exp/lib/python3.11/site-packages/osgeo/gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(
/opt/anaconda3/envs/pop_exp/lib/python3.11/site-packages/popexposure/utils/mask_raster_partial_pixel.py:99: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[61.78839694 73.15033403 66.10942424 60.76798562         nan 52.34760253
 62.74297387 50.9126304  64.69330805 69.37908157 71.03678894 61.68815781
 66.10401055 68.60925908 48.52514369 69.46094513 50.51792842 65.43913195
 69.15735283 46.0351547  54.46632194         nan 38.5443901  68.58695984
 68.12841165 64.25642885 63.08104819 67.27910215 71.74205926 67.49141072
 59.89848456 69.56508636 69.31136322         nan 72.05277966 32.81478683
 63.84865326 68.02154963 62.60602255         nan 64.49564729 61.78146544
 62.51

Processing australia...


/opt/anaconda3/envs/pop_exp/lib/python3.11/site-packages/popexposure/utils/mask_raster_partial_pixel.py:99: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[58.94496284 66.05994066 59.3654089  59.7447079  42.22657883 66.03602642
         nan         nan         nan 64.20215607         nan         nan
 31.93907467 55.1341463  38.9168772          nan 18.57080281 53.29636764
         nan         nan 63.05056763         nan 62.23359729         nan
 52.93705828 64.90275574         nan 64.52299519         nan         nan
         nan 64.13604444         nan         nan         nan 62.64683414
         nan 61.9629453  56.02945244 55.11832428         nan         nan
         nan         nan         nan         nan 59.20041106 61.94488854
 58.71074295 48.19836294 40.31749492 49.05177962         nan         nan
         nan         nan 49.92137011 59.66473237 51.27878613 64.91400909
 72.59811401         nan 55.39922

Processing austria...


...

Processing united_states...


/opt/anaconda3/envs/pop_exp/lib/python3.11/site-packages/popexposure/utils/mask_raster_partial_pixel.py:99: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[61.41726881         nan 60.21853485 ... 60.50575866 60.63632831
 57.68212761]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result.loc[valid_mask] = num_exposed[stat].values.astype(float)
/opt/anaconda3/envs/pop_exp/lib/python3.11/site-packages/popexposure/utils/mask_raster_partial_pixel.py:99: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[59.03881813 60.17559052 59.85399527 ... 60.41915344 60.74068363
 56.8436372 ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result.loc[valid_mask] = num_exposed[stat].values.astype(float)
/opt/anaconda3/envs/pop_exp/lib/python3.11/site-packages/pop

Processing venezuela...


/opt/anaconda3/envs/pop_exp/lib/python3.11/site-packages/popexposure/utils/mask_raster_partial_pixel.py:99: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[52.17096076 80.86359033 65.53705734 71.17688451 76.54946615         nan
 71.81240082         nan 74.78761708 64.49698874         nan         nan
         nan 54.35513859 54.03042695         nan         nan 58.23122918
 54.08638976         nan 58.14199514         nan 61.4920022          nan
 73.85543551 55.14313324 56.23041227 42.10155741 65.4528656  44.62520409
         nan         nan 76.30193434         nan 65.65924227 74.21599037
         nan 73.08291827 58.79904556         nan         nan         nan
 60.09409127         nan]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result.loc[valid_mask] = num_exposed[stat].values.astype(float)
/opt/anaconda3/envs/pop_exp/lib/python3.11/site-packages/popexposure/uti

NameError: name 'combined_df' is not defined